In [1]:
import pandas as pd

data = {
    "날짜": [
        "2025-01-01", "2025-01-02", "2025-01-03",
        "2025-02-01", "2025-02-02", "2025-02-03",
        "2025-03-01", "2025-03-02", "2025-03-03"
    ],
    "매출": [
        100, 120, None,
        200, None, 220,
        300, 320, None
    ]
}

df = pd.DataFrame(data)

df["날짜"] = pd.to_datetime(df["날짜"])

print(df)

          날짜     매출
0 2025-01-01  100.0
1 2025-01-02  120.0
2 2025-01-03    NaN
3 2025-02-01  200.0
4 2025-02-02    NaN
5 2025-02-03  220.0
6 2025-03-01  300.0
7 2025-03-02  320.0
8 2025-03-03    NaN


In [2]:
# ① 월 파생변수 생성 → ② 월별 평균 계산 → ③ 해당 월의 평균으로 결측치 대체
# 이 후에 groupby() 메서드를 배우면 더 간단히 할 수 있다.

df["월"] = df["날짜"].dt.month
print(df)

          날짜     매출  월
0 2025-01-01  100.0  1
1 2025-01-02  120.0  1
2 2025-01-03    NaN  1
3 2025-02-01  200.0  2
4 2025-02-02    NaN  2
5 2025-02-03  220.0  2
6 2025-03-01  300.0  3
7 2025-03-02  320.0  3
8 2025-03-03    NaN  3


In [3]:
jan_mean = df.loc[df["월"] == 1, "매출"].mean()
feb_mean = df.loc[df["월"] == 2, "매출"].mean()
mar_mean = df.loc[df["월"] == 3, "매출"].mean()

print("1월 평균:", jan_mean)
print("2월 평균:", feb_mean)
print("3월 평균:", mar_mean)

1월 평균: 110.0
2월 평균: 210.0
3월 평균: 310.0


In [4]:
df.loc[(df["월"] == 1) & (df["매출"].isna()), "매출"] = jan_mean
df.loc[(df["월"] == 2) & (df["매출"].isna()), "매출"] = feb_mean
df.loc[(df["월"] == 3) & (df["매출"].isna()), "매출"] = mar_mean
df

,날짜,매출,월
0,2025-01-01,100.0,1
1,2025-01-02,120.0,1
2,2025-01-03,110.0,1
3,2025-02-01,200.0,2
4,2025-02-02,210.0,2
5,2025-02-03,220.0,2
6,2025-03-01,300.0,3
7,2025-03-02,320.0,3
8,2025-03-03,310.0,3


In [6]:
cond = (df['월'] == 1)
print(cond)

0     True
1     True
2     True
3    False
4    False
5    False
6    False
7    False
8    False
Name: 월, dtype: bool


In [7]:
df[cond]

,날짜,매출,월
0,2025-01-01,100.0,1
1,2025-01-02,120.0,1
2,2025-01-03,110.0,1


In [8]:
df[df['월'] == 1]['매출']

0    100.0
1    120.0
2    110.0
Name: 매출, dtype: float64

In [9]:
df.loc[df['월'] == 1, '매출']

0    100.0
1    120.0
2    110.0
Name: 매출, dtype: float64

In [10]:
mean1 = (df.loc[df['월'] == 1, '매출']).mean()
mean1

np.float64(110.0)

In [11]:
mean2 = (df.loc[df['월'] == 2, '매출']).mean()
mean2

np.float64(210.0)

In [12]:
mean3 = (df.loc[df['월'] == 3, '매출']).mean()
mean3

np.float64(310.0)

In [ ]:
from sklearn.linear_model import LinearRegression

# 열x, 열y -> y = b + ax
# 열x1, 열x2, 열y -> y = b + ax1 + ax2
# 10 = 5 + 3*1 + 2*1 -> 결측치 -> 1

In [17]:
data = {
    "공부시간": [1, 2, 3, 4, 5, 6, 7, 8],
    "게임시간": [5, 4, 4, 3, 3, 2, 2, 1],
    "시험점수": [45, 50, None, 62, 68, None, 80, 88]
}

df = pd.DataFrame(data)

df

,공부시간,게임시간,시험점수
0,1,5,45.0
1,2,4,50.0
2,3,4,NaN
3,4,3,62.0
4,5,3,68.0
5,6,2,NaN
6,7,2,80.0
7,8,1,88.0


In [18]:
train = df[df["시험점수"].notna()] # 시험점수가 있는 데이터만 학습에 사용

X_train = train[["공부시간", "게임시간"]]
y_train = train["시험점수"]

print(X_train)
print(y_train)

   공부시간  게임시간
0     1     5
1     2     4
3     4     3
4     5     3
6     7     2
7     8     1
0    45.0
1    50.0
3    62.0
4    68.0
6    80.0
7    88.0
Name: 시험점수, dtype: float64


In [20]:
# 회귀모델 학습

from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)

print("회귀계수:", model.coef_)
print("절편:", model.intercept_)

# 시험점수 = 절편 + (공부시간 × 계수1) + (게임시간 × 계수2)
# 시험점수 = (5.78*공부시간) + (-0.60*게임시간) + 41.28

회귀계수: [ 5.78571429 -0.60714286]
절편: 41.28571428571432


In [21]:
missing = df["시험점수"].isna()
print(df[missing])

X_missing = df.loc[missing, ["공부시간", "게임시간"]]
predicted_score = model.predict(X_missing)
print(predicted_score)

   공부시간  게임시간  시험점수
2     3     4   NaN
5     6     2   NaN
[56.21428571 74.78571429]


In [22]:
df.loc[missing, "시험점수"] = [round(x) for x in predicted_score] # 결측치 대체(반올림)
print(df)

   공부시간  게임시간  시험점수
0     1     5  45.0
1     2     4  50.0
2     3     4  56.0
3     4     3  62.0
4     5     3  68.0
5     6     2  75.0
6     7     2  80.0
7     8     1  88.0


In [23]:
model = LinearRegression()
model.fit(X_train, y_train)
print(model.coef_)
print(model.intercept_)

[ 5.78571429 -0.60714286]
41.28571428571432


In [29]:
import numpy as np
df.loc[0, '시험점수'] = np.nan

In [30]:
df.loc[0, '시험점수'] = np.nan

In [31]:
cond = df['시험점수'].isna()
m = df.loc[cond, ['공부시간', '게임시간']]
m

,공부시간,게임시간
0,1,5


In [32]:
result_y = model.predict(m)
print(result_y, type(result_y)) # 내부적 사용하는 numpy가 결과값으로... numpy 학습이 필요하다.

[44.03571429] <class 'numpy.ndarray'>


In [33]:
df.loc[cond, '시험점수'] = result_y.round(1)
df

,공부시간,게임시간,시험점수
0,1,5,44.0
1,2,4,50.0
2,3,4,56.0
3,4,3,62.0
4,5,3,68.0
5,6,2,75.0
6,7,2,80.0
7,8,1,88.0
